# Single LST-Stacked Baseline pI FRF SNR

**by Josh Dillon**, last updated April 29, 2026

This notebook is the LST-stack analog of [single_baseline_pI_FRF_SNR](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_baseline_pI_FRF_SNR.ipynb), specialized for post-LST-binning single-baseline files. It computes delay-filtered and fringe-rate-filtered pseudo-Stokes pI SNR waterfalls suitable for downstream RFI flagging (Round 6).

The key difference from the per-night version: instead of normalizing by the raw autocorrelation (which has heterogeneous spectral and temporal structure post-LST-stack), we first **2D-smooth the autos** with a low delay cutoff and per-channel auto FR halfwidth, with a single round of outlier rejection ("blacklist") and re-fit. This gives a smooth `σ(f)`, so smooth foregrounds remain smooth in the SNR and the delay filter can suppress them cleanly. Per-pixel nsamples in the variance is also replaced with a per-integration band-median.

Configuration is read from a `toml` with reduce passing of environmental variables.

Here's a set of links to skip to particular figures:
# [• Figure 1: Smoothed-Auto Diagnostic](#Figure-1:-Smoothed-Auto-Diagnostic)
# [• Figure 2: Delay-Filtered pI SNR in Fringe Rate Space](#Figure-2:-Delay-Filtered-pI-SNR-in-Fringe-Rate-Space)
# [• Figure 3: Delay-Filtered pI SNR Waterfall](#Figure-3:-Delay-Filtered-pI-SNR-Waterfall)
# [• Figure 4: Delay-Filtered pI SNR Histogram](#Figure-4:-Delay-Filtered-pI-SNR-Histogram)
# [• Figure 5: Delay+FR-Filtered pI SNR Waterfall](#Figure-5:-Delay+FR-Filtered-pI-SNR-Waterfall)
# [• Figure 6: Delay+FR-Filtered pI SNR Histogram](#Figure-6:-Delay+FR-Filtered-pI-SNR-Histogram)


In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
from pathlib import Path
import re
import glob
import copy
import hashlib
import tempfile
import toml
import h5py
import hdf5plugin
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from astropy import units
from scipy import interpolate
from scipy.signal.windows import blackmanharris
from pyuvdata import UVFlag
from hera_cal import io, utils, flag_utils, red_groups
from hera_cal.frf import sky_frates, sky_frates_single, get_FR_buffer_from_spectra, get_m2f_mixer
from hera_filters.dspec import dpss_operator, fourier_filter, sparse_linear_fit_2D
import hera_pspec as hp
import uvtools
from IPython.display import display
%matplotlib inline
_ = np.seterr(all='ignore')
%config InlineBackend.figure_format = 'retina'

In [ ]:
# Path-style settings come in as env vars (set by the bash wrapper).
TOML_FILE: str = os.environ.get('TOML_FILE',
    '/lustre/aoc/projects/hera/h6c-analysis/IDR3/src/hera_pipelines/pipelines/h6c/idr3/v1/post-lstbin/h6c_pspec_16band.toml')
SINGLE_BL_FILE: str = os.environ.get('SINGLE_BL_FILE',
    '/lustre/aoc/projects/hera/h6c-analysis/IDR3/pspec-outputs-131-nights/zen.LST.baseline.0_4.sum.uvh5')
OUT_SNR_FILE: str = os.environ.get('OUT_SNR_FILE', str(Path(SINGLE_BL_FILE).with_suffix('.pI_FRF_SNR.uvh5')))

for setting in ['TOML_FILE', 'SINGLE_BL_FILE', 'OUT_SNR_FILE']:
    print(f'{setting} = "{eval(setting)}"')

# Load all FRF_SNR_OPTS knobs from the toml and inject into globals.
toml_options = toml.load(TOML_FILE)
for toml_section in ['GLOBAL_OPTS', 'FRF_SNR_OPTS']:
    print(f'\nLoading config from [{toml_section}] in {TOML_FILE}:')
    for key, val in toml_options[toml_section].items():
        globals()[key.upper()] = val
        print(f'  {key.upper()} = {val!r}')

SINGLE_BL_FILE = Path(SINGLE_BL_FILE)
FR_SPECTRA_FILE = Path(FR_SPECTRA_FILE)
AUTO_FR_SPECTRUM_FILE = Path(AUTO_FR_SPECTRUM_FILE)
OUT_SNR_FILE = Path(OUT_SNR_FILE)

## Load Data

In [ ]:
# figure out ANTPAIR and the sibling auto file (i==j) via parent-glob
ANTPAIR = tuple(int(a) for a in re.search(r'(\d+)_(\d+)', SINGLE_BL_FILE.name).group().split('_'))
glob_pattern = SINGLE_BL_FILE.parent / SINGLE_BL_FILE.name.replace(f'{ANTPAIR[0]}_{ANTPAIR[1]}', '*').replace('.FR0filt.', '.')
auto_candidates = sorted(SINGLE_BL_FILE.parent.glob(glob_pattern.name))
AUTO_BL_FILE = next(f for f in auto_candidates
                    if (m := re.search(r'(\d+)_(\d+)', f.name))
                    and m.group(1) == m.group(2))
print(f'Loading cross from {SINGLE_BL_FILE.name}')
print(f'Loading auto from  {AUTO_BL_FILE.name}')

# Load both (reading times only from the cross, applied to the auto too)
single_bl_times = np.array(io.HERAData(str(SINGLE_BL_FILE)).times)
hd = io.HERAData([str(SINGLE_BL_FILE), str(AUTO_BL_FILE)])
data, flags, nsamples = hd.read(times=single_bl_times, polarizations=['ee', 'nn'])

# Split into cross + auto views
cross_bls = [ANTPAIR + (pol,) for pol in data.pols()]
auto_antpair = sorted(set([k[:2] for k in data.bls() if k[0] == k[1]]))[0]
auto_bls = [auto_antpair + (pol,) for pol in data.pols()]

dt = np.median(np.diff(data.times)) * 24 * 3600  # seconds
df = np.median(np.diff(data.freqs))               # Hz

print(f'\nantpair = {ANTPAIR}')
print(f'auto antpair = {auto_antpair}')
print(f'dt = {dt:.3f} s, df = {df / 1e6:.4f} MHz')
print(f'data shape = ({len(data.times)}, {len(data.freqs)})')

# Bail early if MIN_SAMP_FRAC isn't met by either polarization
med_auto_n = {pol: np.median(nsamples[auto_antpair + (pol,)]) for pol in ('ee', 'nn')}
SKIP_THIS_FILE = False
if not any(np.median(nsamples[bl]) > MIN_SAMP_FRAC * med_auto_n[bl[2]] for bl in cross_bls):
    print(f'No polarization has nsamples > {MIN_SAMP_FRAC} * auto median; skipping.')
    SKIP_THIS_FILE = True

# Get tslices and bands split around FM (will be reused throughout)
flags_combined = flags[ANTPAIR + ('ee',)] | flags[ANTPAIR + ('nn',)]
tslices_bl, bands_bl = flag_utils.get_minimal_slices(
    flags_combined, freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
print(f'tslices = {tslices_bl}')
print(f'bands = {bands_bl}')

## Helper Functions

In [ ]:
def compute_mb_fr_ranges(dc, antpair):
    '''Compute main beam fringe rate ranges from beam simulation spectra.
    Returns per-frequency arrays: freqs in MHz, FR bounds in mHz.'''
    with h5py.File(FR_SPECTRA_FILE, "r") as h5f:
        metadata = h5f["metadata"]
        bl_to_index_map = {tuple(ap): int(index) for index, antpairs
                           in metadata["baseline_groups"].items() for ap in antpairs}
        spectrum_freqs = metadata["frequencies_MHz"][()] * 1e6
        m_modes = metadata["erh_mode_integer_index"][()]
        this_red_group = red_groups.RedundantGroups.from_antpos(dc.antpos)[antpair]
        for ap in this_red_group:
            if ap in bl_to_index_map:
                mmode_spectrum = h5f["erh_mode_power_spectrum"][:, :, bl_to_index_map[ap]]
                break
            elif ap[::-1] in bl_to_index_map:
                mmode_spectrum = h5f["erh_mode_power_spectrum"][:, :, bl_to_index_map[ap[::-1]]]
                m_modes = m_modes * -1
                break
        else:
            raise KeyError(f'{antpair}, nor any baseline redundant with it, was found in bl_to_index_map.')

    # Build mixing matrix from m-modes to fringe rates
    full_times = dc.times
    times_ks = (full_times - full_times[0] + np.median(np.diff(full_times))) * units.day.to(units.ks)
    filt_frates = np.fft.fftshift(np.fft.fftfreq(times_ks.size, d=np.median(np.diff(times_ks))))
    _m2f_mixer = get_m2f_mixer(times_ks, m_modes)

    # Vectorized: FR spectrum for every spectrum_freq channel at once
    # _m2f_mixer: (n_frates, n_mmodes), mmode_spectrum: (n_mmodes, n_spectrum_freqs)
    fr_spectra = np.abs(np.einsum("fm,mc,mf->fc", _m2f_mixer, mmode_spectrum, _m2f_mixer.T.conj()))
    # Normalize each channel
    fr_spectra /= fr_spectra.sum(axis=0, keepdims=True)

    # Compute quantile bounds per spectrum_freq channel
    cumsum = np.cumsum(fr_spectra, axis=0)
    spec_tops = np.array([np.interp(FR_QUANTILE_HIGH, cumsum[:, c], filt_frates)
                           for c in range(len(spectrum_freqs))])
    spec_bottoms = np.array([np.interp(FR_QUANTILE_LOW, cumsum[:, c], filt_frates)
                              for c in range(len(spectrum_freqs))])

    # Interpolate from spectrum_freqs to data freqs (with extrapolation)
    mb_frate_tops = interpolate.interp1d(spectrum_freqs, spec_tops, fill_value='extrapolate')(dc.freqs)
    mb_frate_bottoms = interpolate.interp1d(spectrum_freqs, spec_bottoms, fill_value='extrapolate')(dc.freqs)

    return (dc.freqs / 1e6, mb_frate_tops, mb_frate_bottoms)


def compute_sky_fr_ranges(dc, antpair, latitude):
    '''Compute sky fringe rate ranges using analytic sky_frates_single + empirical buffer.
    Returns per-frequency arrays in MHz and mHz.'''
    blvec = dc.antpos[antpair[0]] - dc.antpos[antpair[1]]
    sky_centers, sky_hws = sky_frates_single(dc.freqs, blvec, latitude)  # mHz
    fr_buffer = get_FR_buffer_from_spectra(AUTO_FR_SPECTRUM_FILE, dc.times, dc.freqs, gauss_fit_buffer_cut=1e-5)
    return (dc.freqs / 1e6, sky_centers + sky_hws + fr_buffer, sky_centers - sky_hws - fr_buffer)


def overlaps_FR0(bands_bl, mb_frate_tops, mb_frate_bottoms):
    '''Check whether any band's main beam fringe rates overlap FR=0 ± XTALK_FRATE.'''
    for band in bands_bl:
        if band is None:
            continue
        # check if all FRs are above the FR=0 band
        if not ((np.all(mb_frate_tops[band] > XTALK_FRATE)) and (np.all(mb_frate_bottoms[band] > XTALK_FRATE))):
            # check if all FRs are below the FR=0 band
            if not ((np.all(mb_frate_tops[band] < -XTALK_FRATE)) and (np.all(mb_frate_bottoms[band] < -XTALK_FRATE))):
                return True
    return False

## Smooth Autos via 2D DPSS

The autos have non-trivial spectral and temporal structure that, if used directly as σ in `SNR = d/σ`, leaves wiggly residuals that the delay filter can't suppress. We 2D-smooth each polarization's auto in two stages:

1. **Coarse pre-fit at half the main delay (`AUTO_INPAINT_DELAY * AUTO_BLACKLIST_SMOOTHER_FACTOR`) and half the FR halfwidth.** Build a blacklist where the relative error `|raw − coarse| / coarse` exceeds `AUTO_BLACKLIST_REL_ERR_THRESH` (matching the calibration-smoothing approach), then re-fit at the same coarse resolution with those pixels softly downweighted to get a clean, smoother model at the blacklisted pixels.
2. **Final fit at the main delay `AUTO_INPAINT_DELAY` and FR halfwidth.** Blacklisted pixels are replaced by the coarse refit and carry weight `AUTO_BLACKLIST_DOWNWEIGHT` (≈10%); non-blacklisted pixels keep their original data and weights. This gives the DPSS basis a sane value to fit at blacklisted pixels (preventing ringing) while still letting the data drive the fit elsewhere.

The resulting smooth autos serve as the noise-power reference for the SNR computation.

In [ ]:
# Two-stage auto-smoothing knobs
def smooth_one_auto(auto_wf, nsamp_wf, dc, tslices, bands,
                    inpaint_delay_ns=AUTO_INPAINT_DELAY,
                    blacklist_rel_err_thresh=AUTO_BLACKLIST_REL_ERR_THRESH,
                    blacklist_smoother_factor=AUTO_BLACKLIST_SMOOTHER_FACTOR,
                    blacklist_downweight=AUTO_BLACKLIST_DOWNWEIGHT,
                    eigenval_cutoff=EIGENVAL_CUTOFF,
                    cg_tol=AUTO_CG_TOL,
                    cg_iter_lim=AUTO_CG_ITER_LIM):
    '''Fit a 2D DPSS model to one auto polarization in two stages.

    Stage 1 (coarse, at delay = inpaint_delay_ns * blacklist_smoother_factor and FR halfwidth
    scaled by the same factor):
      (a) Fit with the original nsamples weights → coarse_v0.
      (b) Build a blacklist where |raw - coarse_v0| / |coarse_v0| > blacklist_rel_err_thresh.
      (c) Re-fit at the same coarse resolution with blacklisted pixels softly downweighted → coarse_v1.

    Stage 2 (final, at delay = inpaint_delay_ns and the full FR halfwidth):
      Replace the data at blacklisted pixels with coarse_v1, downweight those pixels by
      blacklist_downweight, then fit. Returns (smooth, blacklist).'''
    smooth = np.full_like(auto_wf, np.nan, dtype=complex)
    blacklist = np.zeros(auto_wf.shape, dtype=bool)
    times_s = (dc.times - dc.times[0]) * 86400.0  # seconds
    freqs_hz = dc.freqs

    for tslice, band in zip(tslices, bands):
        if (tslice is None) or (band is None):
            continue

        # Per-band FR halfwidth from the simulated auto FR spectrum (conservative max-over-band).
        fr_buffer = get_FR_buffer_from_spectra(
            str(AUTO_FR_SPECTRUM_FILE), dc.times[tslice], freqs_hz[band],
            gauss_fit_buffer_cut=AUTO_GAUSS_FIT_BUFFER_CUT,
        )
        fr_hw_hz = np.max(fr_buffer) / 1e3

        # Inverse-variance weights for an auto: σ²(V_ii) ∝ V_ii² / nsamples, so weight ∝ nsamples / V_ii².
        with np.errstate(invalid='ignore', divide='ignore'):
            weights = np.where(nsamp_wf[tslice, band] > 0,
                               nsamp_wf[tslice, band] / auto_wf[tslice, band]**2,
                               0).astype(float)
        if not np.any(weights > 0):
            continue

        # ---- Stage 1: coarse pre-fit at scaled delay & FR halfwidth ----
        coarse_freq_basis, _ = dpss_operator(
            freqs_hz[band], [0.0], [inpaint_delay_ns * blacklist_smoother_factor / 1e9],
            eigenval_cutoff=[eigenval_cutoff],
        )
        coarse_time_basis, _ = dpss_operator(
            times_s[tslice], [0.0], [fr_hw_hz * blacklist_smoother_factor],
            eigenval_cutoff=[eigenval_cutoff],
        )
        coarse_coeffs_v0, _ = sparse_linear_fit_2D(
            data=auto_wf[tslice, band],
            weights=weights,
            axis_1_basis=coarse_time_basis,
            axis_2_basis=coarse_freq_basis,
            precondition_solver=True,
            atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        coarse_v0 = np.abs(coarse_time_basis @ coarse_coeffs_v0 @ coarse_freq_basis.T)

        # Blacklist: relative error of raw against coarse fit.
        with np.errstate(invalid='ignore', divide='ignore'):
            rel_err = np.abs(auto_wf[tslice, band] - coarse_v0) / np.abs(coarse_v0)
        bl_local = (np.isfinite(rel_err) & (rel_err > blacklist_rel_err_thresh)
                    & (nsamp_wf[tslice, band] > 0))
        blacklist[tslice, band] = bl_local

        # Coarse re-fit: same basis, but downweight the blacklisted pixels so we get a clean smoother model
        # we can use to fill them in for the final fit.
        weights_coarse_v1 = weights * np.where(bl_local, blacklist_downweight, 1.0)
        if not np.any(weights_coarse_v1 > 0):
            smooth[tslice, band] = coarse_v0
            continue
        coarse_coeffs_v1, _ = sparse_linear_fit_2D(
            data=auto_wf[tslice, band],
            weights=weights_coarse_v1,
            axis_1_basis=coarse_time_basis,
            axis_2_basis=coarse_freq_basis,
            precondition_solver=True,
            atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        coarse_v1 = np.abs(coarse_time_basis @ coarse_coeffs_v1 @ coarse_freq_basis.T)

        # ---- Stage 2: final fit at full delay & FR halfwidth ----
        # Plug coarse_v1 in at blacklisted pixels and downweight them so they're fit but not trusted.
        data_final = np.where(bl_local, coarse_v1, auto_wf[tslice, band])
        weights_final = weights * np.where(bl_local, blacklist_downweight, 1.0)
        freq_basis, _ = dpss_operator(
            freqs_hz[band], [0.0], [inpaint_delay_ns / 1e9],
            eigenval_cutoff=[eigenval_cutoff],
        )
        time_basis, _ = dpss_operator(
            times_s[tslice], [0.0], [fr_hw_hz],
            eigenval_cutoff=[eigenval_cutoff],
        )
        final_coeffs, _ = sparse_linear_fit_2D(
            data=data_final,
            weights=weights_final,
            axis_1_basis=time_basis,
            axis_2_basis=freq_basis,
            precondition_solver=True,
            atol=cg_tol, btol=cg_tol, iter_lim=cg_iter_lim,
        )
        smooth[tslice, band] = np.abs(time_basis @ final_coeffs @ freq_basis.T)

    return smooth, blacklist

if not SKIP_THIS_FILE:
    # Cache key: TOML + auto file basename. Mirrors single_baseline_lst_stack_and_reinpaint.ipynb cell 11.
    cache_key = hashlib.md5((toml.dumps(toml_options) + AUTO_BL_FILE.name).encode()).hexdigest()
    auto_cache_file = OUT_SNR_FILE.parent / f'smooth_autos_cache_{cache_key}.npz'

    auto_smooth = {}
    auto_blacklist = {}
    if auto_cache_file.exists():
        print(f'Loading cached smoothed autos from {auto_cache_file}')
        with np.load(auto_cache_file) as cache:
            assert cache['toml_hash'].item() == cache_key, 'Cache hash mismatch'
            for pol in ('ee', 'nn'):
                auto_smooth[pol] = cache[f'auto_smooth_{pol}']
                auto_blacklist[pol] = cache[f'auto_blacklist_{pol}']
    else:
        print(f'No cache found at {auto_cache_file}, smoothing autos...')
        for pol in ('ee', 'nn'):
            bl = auto_antpair + (pol,)
            print(f'Smoothing auto {bl}...')
            smooth, blk = smooth_one_auto(np.abs(data[bl]), nsamples[bl], data, tslices_bl, bands_bl)
            auto_smooth[pol] = smooth
            auto_blacklist[pol] = blk
            n_bl = int(np.sum(blk))
            n_unflagged = int(np.sum(nsamples[bl] > 0))
            if n_unflagged > 0:
                print(f'  Blacklisted {n_bl} of {n_unflagged} unflagged pixels '
                    f'({100 * n_bl / n_unflagged:.2f}%) at rel_err > {AUTO_BLACKLIST_REL_ERR_THRESH}.')

        # Atomic write to handle parallel race conditions (parallel baselines share the auto cache).
        tmp_fd, tmp_path = tempfile.mkstemp(dir=str(OUT_SNR_FILE.parent), suffix='.npz')
        os.close(tmp_fd)
        np.savez(tmp_path,
                toml_hash=cache_key,
                **{f'auto_smooth_{pol}': auto_smooth[pol] for pol in ('ee', 'nn')},
                **{f'auto_blacklist_{pol}': auto_blacklist[pol] for pol in ('ee', 'nn')})
        try:
            os.rename(tmp_path, auto_cache_file)
            print(f'Saved smoothed autos cache to {auto_cache_file}')
        except OSError:
            os.remove(tmp_path)  # another process beat us to it
            print('Cache already written by another process.')

# *Figure 1: Smoothed-Auto Diagnostic*

Per polarization: raw auto magnitude, smoothed auto magnitude, residual / σ_pred, and the blacklist mask.

In [ ]:
def plot_smoothed_auto(pol):
    bl = auto_antpair + (pol,)
    raw = np.abs(data[bl])
    smooth = np.abs(auto_smooth[pol])
    blk = auto_blacklist[pol]
    with np.errstate(invalid='ignore', divide='ignore'):
        rel_err = np.abs(raw - smooth) / smooth
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6,
              data.times[-1] - int(data.times[0]), data.times[0] - int(data.times[0])]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=120, sharex=True, sharey=True)
    im0 = axes[0].imshow(raw, aspect='auto', cmap='viridis', extent=extent, interpolation='none', norm=matplotlib.colors.LogNorm())
    axes[0].set_title(f'{pol} raw |auto|')
    plt.colorbar(im0, ax=axes[0])
    im1 = axes[1].imshow(smooth, aspect='auto', cmap='viridis', extent=extent, interpolation='none',
                         norm=matplotlib.colors.LogNorm(vmin=im0.get_clim()[0], vmax=im0.get_clim()[1]))
    axes[1].set_title(f'{pol} smooth |auto|')
    plt.colorbar(im1, ax=axes[1])
    im2 = axes[2].imshow(rel_err, aspect='auto', cmap='inferno', extent=extent, interpolation='none',
                         vmin=0, vmax=2 * AUTO_BLACKLIST_REL_ERR_THRESH)
    axes[2].set_title(f'{pol} |raw - smooth| / smooth')
    plt.colorbar(im2, ax=axes[2], extend='max')
    im3 = axes[3].imshow(blk.astype(int), aspect='auto', cmap='gray_r', extent=extent, interpolation='none',
                         vmin=0, vmax=1)
    axes[3].set_title(f'{pol} blacklist (rel_err > {AUTO_BLACKLIST_REL_ERR_THRESH})')
    plt.colorbar(im3, ax=axes[3])
    for ax in axes:
        ax.set_xlabel('Frequency (MHz)')
    axes[0].set_ylabel(f'JD - {int(data.times[0])}')
    plt.tight_layout()
    plt.show()

if not SKIP_THIS_FILE:
    for pol in ('ee', 'nn'):
        plot_smoothed_auto(pol)

## Compute Pseudo-Stokes pI SNR

`var_pI = |a_smooth_ee|² / (dt·df) / <n_ee>_band + |a_smooth_nn|² / (dt·df) / <n_nn>_band` — using the band-median nsamples per integration (per polarization, per band, excluding zeros). This makes σ(f) smooth in frequency, so smooth foregrounds in `d_pI` remain smooth in `SNR = d_pI / sqrt(var_pI)` and the binary-weights delay filter can find them.

In [ ]:
# Compute FR ranges for the cross baseline (used by the FR=0 notch and main-beam FR filter)
if not SKIP_THIS_FILE:
    print('Computing fringe rate ranges...')
    mb_frate_freqs_MHz, mb_frate_tops, mb_frate_bottoms = compute_mb_fr_ranges(data, ANTPAIR)
    sky_frate_freqs_MHz, sky_frate_tops, sky_frate_bottoms = compute_sky_fr_ranges(data, ANTPAIR, hd.telescope.location.lat.rad)

    # Optionally skip baselines whose main-beam FR overlaps FR=0 (per the per-night notebook's logic)
    fr0_overlap = np.any((mb_frate_bottoms < XTALK_FRATE) & (mb_frate_tops > -XTALK_FRATE))
    if fr0_overlap and SKIP_FR0_OVERLAP_BASELINES:
        print(f'{ANTPAIR} main-beam FR overlaps FR=0; skipping (set SKIP_FR0_OVERLAP_BASELINES=False to process).')
        SKIP_THIS_FILE = True

if not SKIP_THIS_FILE:
    # Allocate full-waterfall arrays
    filt_flags_full = np.ones((len(data.times), len(data.freqs)), dtype=bool)
    newly_flagged = np.zeros((len(data.times), len(data.freqs)), dtype=bool)
    dly_filt_SNR_full = np.full((len(data.times), len(data.freqs)), np.nan, dtype=complex)
    frf_SNR_full = np.full((len(data.times), len(data.freqs)), np.nan, dtype=complex)
    taper_2d = np.zeros((len(data.times), len(data.freqs)))
    fully_inpainted = np.zeros((len(data.times), len(data.freqs)), dtype=bool)

    for tslice, band in zip(tslices_bl, bands_bl):
        if (band is None) or np.all(flags_combined[tslice, band]):
            continue

        d_ee = data[ANTPAIR + ('ee',)][tslice, band]
        d_nn = data[ANTPAIR + ('nn',)][tslice, band]
        f_ee = flags[ANTPAIR + ('ee',)][tslice, band]
        f_nn = flags[ANTPAIR + ('nn',)][tslice, band]
        n_ee = nsamples[ANTPAIR + ('ee',)][tslice, band]
        n_nn = nsamples[ANTPAIR + ('nn',)][tslice, band]
        a_smooth_ee = np.abs(auto_smooth['ee'][tslice, band])
        a_smooth_nn = np.abs(auto_smooth['nn'][tslice, band])

        # keep track of pixels 100% inpainted in either ee or nn
        fully_inpainted[tslice, band] |= ((n_ee == 0) | (n_nn == 0))

        # Band-median nsamples per integration (per pol, excluding zeros)
        with np.errstate(invalid='ignore'):
            n_ee_eff = np.nanmedian(np.where(n_ee > 0, n_ee, np.nan), axis=1, keepdims=True)
            n_nn_eff = np.nanmedian(np.where(n_nn > 0, n_nn, np.nan), axis=1, keepdims=True)
        n_ee_eff = np.broadcast_to(n_ee_eff, n_ee.shape)
        n_nn_eff = np.broadcast_to(n_nn_eff, n_nn.shape)

        # Variance using smoothed autos and band-median nsamples
        var_pI = a_smooth_ee**2 / (dt * df) / n_ee_eff + a_smooth_nn**2 / (dt * df) / n_nn_eff

        # Form pseudo-Stokes pI
        d_pI, f_pI, _ = hp.pstokes._combine_pol_arrays(
            'ee', 'nn', 'pI', pol_convention=hd.pol_convention,
            data_list=[d_ee, d_nn], flags_list=[f_ee, f_nn],
            nsamples_list=[n_ee, n_nn],
            x_orientation=hd.telescope.get_x_orientation_from_feeds())
        d_pI[f_pI] = 0
        SNR = d_pI / var_pI**.5
        f_pI |= ~np.isfinite(SNR)
        SNR[f_pI] = 0

        # Stash for the next cell to delay-filter
        if 'snr_band_data' not in dir():
            snr_band_data = []
        snr_band_data.append(dict(
            tslice=tslice, band=band, SNR=SNR, f_pI=f_pI,
        ))

        # Build per-band 2D taper for plotting
        band_ntimes = tslice.stop - tslice.start
        taper_2d[tslice, band] = blackmanharris(band_ntimes)[:, np.newaxis]

    print(f'Computed SNR for {len(snr_band_data)} band(s).')

## Delay Filter + FR=0 Notch + FR Filter

Same math as the per-night SNR notebook (binary weights on the SNR, leverage correction, per-channel FR=0 notch where main beam overlaps, then per-channel main-beam FR filter). The smoothed σ is what makes this work cleanly here.

In [ ]:
if not SKIP_THIS_FILE:
    for sb in snr_band_data:
        tslice, band = sb['tslice'], sb['band']
        SNR = sb['SNR']
        f_pI = sb['f_pI']
        f_pI_or_inpaint = f_pI | fully_inpainted[tslice, band]

        # ---- Delay filter (dpss_matrix, per-integration leverage correction) ----
        print(f'\tDelay filtering band {band}...')
        wgts = np.where(f_pI_or_inpaint, 0, 1).astype(float)
        filter_kwargs = dict(filter_centers=[0], filter_half_widths=[FILTER_DELAY * 1e-9],
                            eigenval_cutoff=[EIGENVAL_CUTOFF], suppression_factors=[EIGENVAL_CUTOFF],
                            max_contiguous_edge_flags=len(data.freqs))
        try:
            result, _, info = fourier_filter(data.freqs[band], SNR, wgts=wgts,
                                            mode='dpss_matrix', **filter_kwargs)
        except np.linalg.LinAlgError as e:
            from hera_filters.dspec import dpss_operator as _dop
            X = _dop(data.freqs[band], [0],
                     filter_half_widths=[FILTER_DELAY * 1e-9],
                     eigenval_cutoff=[EIGENVAL_CUTOFF])[0]
            print(f'  ANTPAIR={ANTPAIR} band={band}')
            print(f'  SNR finite={np.all(np.isfinite(SNR))} #nan={np.sum(np.isnan(SNR))} #inf={np.sum(np.isinf(SNR))}')
            print(f'  wgts finite={np.all(np.isfinite(wgts))}  freqs finite={np.all(np.isfinite(data.freqs[band]))}')
            print(f'  X finite={np.all(np.isfinite(X))}  X.shape={X.shape}')
            from pathlib import Path
            _dump_path = str(Path(SINGLE_BL_FILE).parent / f'svd_fail_{ANTPAIR[0]}_{ANTPAIR[1]}.npz')
            np.savez(_dump_path, SNR=SNR, wgts=wgts, X=X, freqs=data.freqs[band])
            print(f'  dumped {_dump_path}')
            raise
        dly_filt_SNR = SNR - result

        # Per-integration weighted leverage correction
        X = dpss_operator(data.freqs[band], [0],
                        filter_half_widths=[FILTER_DELAY / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])[0]
        correction_cache = {}
        for i in range(SNR.shape[0]):
            w = wgts[i]
            if not np.any(w):
                continue
            cache_key = f_pI_or_inpaint[i].tobytes()
            if cache_key not in correction_cache:
                XtWX = (X.T * w) @ X
                if np.all(np.isclose(XtWX.imag, 0)):
                    XtWX = np.real(XtWX)
                try:
                    P = np.linalg.pinv(XtWX) @ (X.T * w)
                    lev = np.real(np.sum(X * P.T, axis=1))
                    correction = (1 - lev)**.5
                    correction_cache[cache_key] = np.where(np.isfinite(correction) & (lev > 0) & (lev < 1), correction, np.nan)
                except np.linalg.LinAlgError:
                    correction_cache[cache_key] = np.full(X.shape[0], np.nan)
            dly_filt_SNR[i] /= correction_cache[cache_key]

        # ---- FR=0 notch on dly_filt_SNR, per-channel ----
        # The cache stores (correction_per_t, P_fr0). P_fr0 is reused below in the
        # main FR filter to compute the leverage of the composed operator.
        print(f'\tFR=0 notch filtering band {band}...')
        fr0_halfwidth_hz = XTALK_FRATE / 1000.0  # mHz -> Hz
        times_in_seconds = (data.times[tslice] - data.times[tslice][0]) * 24 * 3600
        Xt_fr0 = dpss_operator(data.times[tslice] * 24 * 3600, filter_centers=[0],
                            filter_half_widths=[fr0_halfwidth_hz], eigenval_cutoff=[EIGENVAL_CUTOFF])[0]
        fr0_correction_cache = {}
        for chan, (fr_low, fr_high) in enumerate(zip(mb_frate_bottoms[band], mb_frate_tops[band])):
            w_fr0 = np.where(f_pI_or_inpaint[:, chan], 0, 1).astype(float)
            if not np.any(w_fr0):
                continue
            cache_key_fr0 = f_pI_or_inpaint[:, chan].tobytes()
            if cache_key_fr0 not in fr0_correction_cache:
                XtWXt_fr0 = np.dot(Xt_fr0.conj().T * w_fr0, Xt_fr0)
                try:
                    P_fr0 = np.linalg.pinv(XtWXt_fr0) @ (Xt_fr0.conj().T * w_fr0)
                    lev_fr0 = np.real(np.sum(Xt_fr0 * P_fr0.T, axis=1))
                    correction_fr0 = (1 - lev_fr0)**.5
                    correction_fr0 = np.where(
                        np.isfinite(correction_fr0) & (lev_fr0 > 0) & (lev_fr0 < 1),
                        correction_fr0, np.nan)
                    fr0_correction_cache[cache_key_fr0] = (correction_fr0, P_fr0)
                except np.linalg.LinAlgError:
                    fr0_correction_cache[cache_key_fr0] = (np.full(Xt_fr0.shape[0], np.nan), None)
            result_fr0, _, _ = fourier_filter(times_in_seconds,
                                            dly_filt_SNR[:, chan:chan+1],
                                            wgts=np.where(f_pI_or_inpaint[:, chan:chan+1], 0, 1),
                                            filter_centers=[0], filter_half_widths=[fr0_halfwidth_hz],
                                            mode='dpss_leastsq',
                                            eigenval_cutoff=[EIGENVAL_CUTOFF],
                                            suppression_factors=[EIGENVAL_CUTOFF],
                                            max_contiguous_edge_flags=len(data.times[tslice]),
                                            filter_dims=0)
            dly_filt_SNR[:, chan:chan+1] -= result_fr0
            dly_filt_SNR[:, chan] /= fr0_correction_cache[cache_key_fr0][0]

        # Update larger arrays
        newly_flagged[tslice, band] |= ~np.isfinite(dly_filt_SNR) & ~f_pI
        f_pI |= newly_flagged[tslice, band]
        dly_filt_SNR[newly_flagged[tslice, band]] = 0
        filt_flags_full[tslice, band] = f_pI
        dly_filt_SNR_full[tslice, band] = dly_filt_SNR

        # ---- Per-channel FR filter at the main-beam center ----
        print(f'\tFR filtering band {band}...')
        frf_SNR = copy.deepcopy(dly_filt_SNR)
        for chan, (fr_low, fr_high) in enumerate(zip(mb_frate_bottoms[band], mb_frate_tops[band])):
            fr_center = (fr_low + fr_high) / 2 / 1000.0  # mHz -> Hz
            fr_halfwidth = (fr_high - fr_low) / 2 / 1000.0

            result, _, _ = fourier_filter(times_in_seconds, dly_filt_SNR[:, chan:chan+1],
                                        wgts=np.where(f_pI_or_inpaint[:, chan:chan+1], 0, 1),
                                        filter_centers=[fr_center],
                                        filter_half_widths=[fr_halfwidth],
                                        mode='dpss_leastsq',
                                        eigenval_cutoff=[EIGENVAL_CUTOFF],
                                        suppression_factors=[EIGENVAL_CUTOFF],
                                        max_contiguous_edge_flags=len(data.times),
                                        filter_dims=0)
            Xt = dpss_operator(data.times[tslice] * 24 * 3600,
                            filter_centers=[fr_center],
                            filter_half_widths=[fr_halfwidth],
                            eigenval_cutoff=[EIGENVAL_CUTOFF])[0]
            W = np.where(f_pI_or_inpaint[:, chan], 0, 1).astype(float)
            XtWXt = np.dot(Xt.conj().T * W, Xt)
            try:
                P = np.linalg.pinv(XtWXt) @ (Xt.conj().T * W)

                # Per-time noise variance after the composed operator
                #   M = H_main @ D_fr0 @ (I - H_fr0)
                # acts on raw unit-variance noise. (H_fr0 is the FR=0 hat matrix
                # already applied above; D_fr0 = diag(1/(1-lev_fr0)**.5) is the
                # per-time rescaling the FR=0 notch step already applied.) Using
                # only h_main_tt over-corrects in channels where the main-beam
                # DPSS subspace overlaps the FR=0 subspace.
                #
                # Factored form: with H_main = Xt @ P and H_fr0 = Xt_fr0 @ P_fr0,
                #   var_t = (Xt @ S @ Xt^H)_tt,  S = Q @ Q^H,
                #   Q = P*D_fr0  -  (P*D_fr0 @ Xt_fr0) @ P_fr0
                # avoids ever materialising the n_t x n_t hat matrices.
                cache_entry = fr0_correction_cache.get(f_pI_or_inpaint[:, chan].tobytes())
                if cache_entry is not None and cache_entry[1] is not None:
                    correction_fr0, P_fr0 = cache_entry
                    D_fr0 = np.where(np.isfinite(correction_fr0), 1.0 / correction_fr0, 0.0)
                    PD = P * D_fr0[None, :]
                    Q = PD - (PD @ Xt_fr0) @ P_fr0
                else:
                    Q = P
                S = Q @ Q.conj().T
                var_t = np.real(np.einsum('tb,bc,tc->t', Xt, S, Xt.conj()))
                lev_t_correction = np.where(var_t > 0, np.sqrt(var_t), np.nan)
            except np.linalg.LinAlgError:
                lev_t_correction = np.full(Xt.shape[0], np.nan)

            frf_SNR[:, chan:chan+1] = np.where(f_pI[:, chan:chan+1], 0, result / lev_t_correction[:, None])

        frf_SNR_full[tslice, band] = frf_SNR
        newly_flagged[tslice, band] |= ~np.isfinite(frf_SNR) & ~f_pI

    if np.any(newly_flagged):
        print(f'\tFlagging {np.sum(newly_flagged)} pixels due to non-finite SNR (degenerate leverage).')
        filt_flags_full |= newly_flagged

## Plot Helpers

In [ ]:
def plot_fr_waterfall(snr_wf, flags_wf, taper_2d, freqs, times, title,
                      mb_frate_freqs_MHz=None, mb_frate_tops=None, mb_frate_bottoms=None,
                      sky_frate_freqs_MHz=None, sky_frate_tops=None, sky_frate_bottoms=None,
                      vmax=5):
    '''Plot freq vs fringe rate waterfall of |SNR| after FFT along time axis.
    Accepts pre-assembled full waterfalls with a 2D per-band taper.'''
    ntimes = len(times)
    times_in_seconds = (times - times[0]) * 24 * 3600
    frates = uvtools.utils.fourier_freqs(times_in_seconds) * 1000  # mHz

    # Per-column normalization accounting for taper and flags
    unflagged = (~flags_wf).astype(float)
    norm = (ntimes * np.mean((taper_2d * unflagged)**2, axis=0))**.5

    # FFT with per-band taper 
    to_plot = np.fft.fftshift(np.fft.fft(taper_2d * np.where((~flags_wf) & (taper_2d > 0), snr_wf, 0), axis=0), axes=0)
    to_plot = np.abs(to_plot) / norm[np.newaxis, :]

    fig = plt.figure(figsize=(14, 8), dpi=200)
    extent = [freqs[0] / 1e6, freqs[-1] / 1e6, frates[-1], frates[0]]
    im = plt.imshow(to_plot, aspect='auto', interpolation='none',
                    extent=extent, vmin=0, vmax=vmax, cmap='plasma')
    plt.colorbar(im, extend='max', label='|pI SNR|')
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Fringe Rate (mHz)')
    plt.title(title)

    if sky_frate_freqs_MHz is not None:
        plt.plot(sky_frate_freqs_MHz, sky_frate_tops, 'w:', lw=1, label='Sky FRs')
        plt.plot(sky_frate_freqs_MHz, sky_frate_bottoms, 'w:', lw=1)
        plt.ylim([-np.max([np.abs(sky_frate_tops), np.abs(sky_frate_bottoms)]) * 1.25,
                  np.max([np.abs(sky_frate_tops), np.abs(sky_frate_bottoms)]) * 1.25])
    else:
        plt.ylim([-5, 5])
    if mb_frate_freqs_MHz is not None:
        plt.plot(mb_frate_freqs_MHz, mb_frate_tops, 'w--', lw=1, label='Main Beam FRs')
        plt.plot(mb_frate_freqs_MHz, mb_frate_bottoms, 'w--', lw=1)
    if sky_frate_freqs_MHz is not None or mb_frate_freqs_MHz is not None:
        plt.legend()

    plt.tight_layout()
    plt.close(fig)
    return fig


def plot_time_freq_waterfall(snr_wf, flags_wf, freqs, times, lsts, title, vmax=5):
    '''Plot freq vs time waterfall of |SNR| in real space with LST right axis.
    Accepts pre-assembled full waterfalls.'''
    to_plot = np.where(flags_wf, np.nan, np.abs(snr_wf))

    fig, ax = plt.subplots(figsize=(14, 8), dpi=200)
    extent = [freqs[0] / 1e6, freqs[-1] / 1e6,
              times[-1] - int(times[0]), times[0] - int(times[0])]
    im = ax.imshow(to_plot, aspect='auto', interpolation='none',
                   extent=extent, vmin=0, vmax=vmax, cmap='plasma')
    plt.colorbar(im, extend='max', label='|pI SNR|', ax=ax)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel(f'JD - {int(times[0])}')
    ax.set_title(title)

    # Add LST right axis with proper wrapping
    lst_grid = lsts * 12 / np.pi  # radians to hours
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    ax2 = ax.twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    plt.tight_layout()
    plt.close(fig)
    return fig


def plot_snr_histograms(snr_wf, flags_wf, title):
    '''Plot histogram of |SNR| compared to the Rayleigh distribution expected for noise-only.
    Accepts pre-assembled full waterfall and flags.'''
    fig = plt.figure(figsize=(12, 5))
    bins = np.arange(0, 10, .01)
    to_hist = np.abs(snr_wf[~flags_wf])
    to_hist = to_hist[np.isfinite(to_hist) & (to_hist > 0)]
    hist = plt.hist(to_hist, bins=bins, density=True, label='Real-space |pI SNR|')
    plt.plot(bins, 2 * bins * np.exp(-bins**2), 'k--', label='Rayleigh Distribution (Noise-Only)')
    plt.yscale('log')
    all_densities = hist[0][hist[0] > 0]
    if len(all_densities) > 0:
        plt.ylim(np.min(all_densities) / 2, np.max(all_densities) * 2)
    plt.legend()
    plt.ylabel('Density')
    plt.xlabel('|pI SNR|')
    plt.xlim([-.5, 10])
    plt.title(title)
    plt.tight_layout()
    plt.close(fig)
    return fig

# *Figure 2: Delay-Filtered pI SNR in Fringe Rate Space*

In [ ]:
if not SKIP_THIS_FILE and not np.all(filt_flags_full):
    fig = plot_fr_waterfall(
        dly_filt_SNR_full, filt_flags_full, taper_2d, data.freqs, data.times,
        f'{ANTPAIR} Delay-Filtered pI',
        mb_frate_freqs_MHz=mb_frate_freqs_MHz, mb_frate_tops=mb_frate_tops, mb_frate_bottoms=mb_frate_bottoms,
        sky_frate_freqs_MHz=sky_frate_freqs_MHz, sky_frate_tops=sky_frate_tops, sky_frate_bottoms=sky_frate_bottoms)
    display(fig)

# *Figure 3: Delay-Filtered pI SNR Waterfall*

In [ ]:
if not SKIP_THIS_FILE and not np.all(filt_flags_full):
    fig = plot_time_freq_waterfall(
        dly_filt_SNR_full, filt_flags_full, data.freqs, data.times, data.lsts,
        f'{ANTPAIR} Delay-Filtered pI')
    display(fig)

# *Figure 4: Delay-Filtered pI SNR Histogram*

In [ ]:
if not SKIP_THIS_FILE and not np.all(filt_flags_full):
    fig = plot_snr_histograms(dly_filt_SNR_full, filt_flags_full, f'{ANTPAIR} Delay-Filtered pI')
    display(fig)

# *Figure 5: Delay+FR-Filtered pI SNR Waterfall*

In [ ]:
if not SKIP_THIS_FILE and not np.all(filt_flags_full):
    fig = plot_time_freq_waterfall(
        frf_SNR_full, filt_flags_full, data.freqs, data.times, data.lsts,
        f'{ANTPAIR} Delay+FR-Filtered pI')
    display(fig)

# *Figure 6: Delay+FR-Filtered pI SNR Histogram*

In [ ]:
if not SKIP_THIS_FILE and not np.all(filt_flags_full):
    fig = plot_snr_histograms(frf_SNR_full, filt_flags_full, f'{ANTPAIR} Delay+FR-Filtered pI')
    display(fig)

## Save SNR uvh5

In [ ]:
if not SKIP_THIS_FILE and not np.all(filt_flags_full):
    # Restrict hd to the cross + ee polarization, then overwrite with FRF SNR labelled as pI
    hd_out = io.HERAData(str(SINGLE_BL_FILE))
    data_out, flags_out, _ = hd_out.read(polarizations=['ee'])
    hd_out.polarization_array[0] = utils.polstr2num('pI')
    bl_out = ANTPAIR + ('ee',)
    data_out[bl_out] = np.where(np.isfinite(frf_SNR_full), frf_SNR_full, 0)
    flags_out[bl_out] = filt_flags_full
    hd_out.update(data=data_out, flags=flags_out)
    print(f'Writing FRF SNR results to {OUT_SNR_FILE}')
    hd_out.write_uvh5(str(OUT_SNR_FILE), clobber=True)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'hera_pspec', 'pyuvdata', 'numpy']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')